# Mephisto Health System: Conflict-of-Interest Detection Pipeline

### Big Data & Cloud Platforms Capstone · Apache Spark + Delta Lake on Databricks

**Source:** CMS Open Payments, General Payments, Program Year 2025 (`bronze/clinicians_pgyr2025.parquet` in S3 - 16,095,794 rows · 16 cols · ZSTD Parquet)

**Scope:** U.S. covered-recipient physicians + non-physician practitioners (teaching hospitals excluded)

**Architecture (1:1 with the diagram):** Bronze (S3) → Silver (clean) → Gold (per-clinician features) → three workloads: Batch KPIs · Structured Streaming · MLlib anomaly detection → Model deployment (Mephisto roster).

> **About the saved outputs.** This is the notebook as last saved in Databricks. Outputs that printed clinician Profile IDs or NPIs have been removed, because Open Payments data names real people. Outputs from cells that changed after the final fix (cluster names in Stage 8.3b and the scored roster in Stage 9) have been cleared, so they don't show the old, wrong labels. The final figures are in the report (`docs/report.pdf`).


In [0]:
BUCKET = "s3://<your-bucket>"   # replace with your own S3 bucket

bronze_path = f"{BUCKET}/bronze/clinicians_pgyr2025.parquet"
silver_path = f"{BUCKET}/silver/clinicians_silver"    # Delta, written in step 2
gold_path   = f"{BUCKET}/gold/clinician_features"      # Delta, written in step 5


## Step 1 - Bronze verification read
confirm the read succeeds, the row/column counts match what was uploaded and inspect the schema 

In [0]:
# Read parquet int a spark df (lazy  no work until an action runs)
bronze_df = spark.read.parquet(bronze_path)
#sanity checks
print(f"bronze_df has {bronze_df.count()} rows")
print(f"bronze_df has {len(bronze_df.columns)} columns")


bronze_df has 16095794 rows
bronze_df has 16 columns


In [0]:
#Exact column and inferred types
bronze_df.printSchema()
display(bronze_df.limit(10))

root
 |-- Record_ID: long (nullable = true)
 |-- Covered_Recipient_Type: string (nullable = true)
 |-- Covered_Recipient_Profile_ID: long (nullable = true)
 |-- Covered_Recipient_NPI: long (nullable = true)
 |-- Covered_Recipient_Specialty_1: string (nullable = true)
 |-- Recipient_State: string (nullable = true)
 |-- Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID: long (nullable = true)
 |-- Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name: string (nullable = true)
 |-- Total_Amount_of_Payment_USDollars: double (nullable = true)
 |-- Date_of_Payment: date (nullable = true)
 |-- Number_of_Payments_Included_in_Total_Amount: long (nullable = true)
 |-- Form_of_Payment_or_Transfer_of_Value: string (nullable = true)
 |-- Nature_of_Payment_or_Transfer_of_Value: string (nullable = true)
 |-- Indicate_Drug_or_Biological_or_Device_or_Medical_Supply_1: string (nullable = true)
 |-- Product_Category_or_Therapeutic_Area_1: string (nullable = true)
 |-- Program_Year: long

## Step 2 - Silver layer (clean)

Filter unusable rows, normalise text, split the pipe-delimited specialty, drop duplicate `Record_ID`s, write Delta.

In [0]:
# Null profiling broze- count nulls per column before cleaning
from pyspark.sql import functions as F

total = bronze_df.count()
null_counts = bronze_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in bronze_df.columns
]).collect()[0].asDict()

print(f"{'Column':<60} {'Nulls':>12} {'%':>7}")
print("-" * 82)
for col, n in sorted(null_counts.items(), key=lambda x: -x[1]):
    print(f"{col:<60} {n:>12,} {n/total*100:>6.2f}%")


Column                                                              Nulls       %
----------------------------------------------------------------------------------
Product_Category_or_Therapeutic_Area_1                          1,292,811   8.03%
Indicate_Drug_or_Biological_or_Device_or_Medical_Supply_1       1,281,269   7.96%
Covered_Recipient_NPI                                               7,238   0.04%
Record_ID                                                               0   0.00%
Covered_Recipient_Type                                                  0   0.00%
Covered_Recipient_Profile_ID                                            0   0.00%
Covered_Recipient_Specialty_1                                           0   0.00%
Recipient_State                                                         0   0.00%
Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID             0   0.00%
Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name            0   0.00%
Total_Amount_o

In [0]:
silver_df = bronze_df

# 1. Drop rows we cant attribute or aggregate:
#   - no Profile_ID - cant tie payment to a clinician
#   -null/negative amount - not a usable payment

silver_df = silver_df.filter(
    F.col("Covered_Recipient_Profile_ID").isNotNull() &
    F.col("Total_Amount_of_Payment_USDollars").isNotNull() &
    (F.col("Total_Amount_of_Payment_USDollars") > 0)
)


In [0]:
# 2. Trim whitespace on text columns; uppercase state for clean grouping
silver_df = (
    silver_df
    .withColumn("Recipient_State", F.upper(F.trim(F.col("Recipient_State"))))
    .withColumn("Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name",
                F.trim(F.col("Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name")))
    .withColumn("Nature_of_Payment_or_Transfer_of_Value",
                F.trim(F.col("Nature_of_Payment_or_Transfer_of_Value")))
)

In [0]:
# 3. Split "Allopathic & Osteopathic Physicians|Radiology|..." on the pipe.
#    specialty_group = broad provider type (element 0)
#    specialty       = specific specialty (element 1, fallback to element 0)
split_col = F.split(F.col("Covered_Recipient_Specialty_1"), r"\|")

silver_df = (
    silver_df
    .withColumn("specialty_group", F.trim(split_col.getItem(0)))
    .withColumn("specialty",
                F.coalesce(F.trim(split_col.getItem(1)), F.trim(split_col.getItem(0))))
)

In [0]:
# 4. Record_ID is the natural primary key — drop any exact duplicates
silver_df = silver_df.dropDuplicates(["Record_ID"])

In [0]:
# Write silver as Delta (overwrite so re-runs are idempotent)
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path))

# Verify
silver_check = spark.read.format("delta").load(silver_path)
print(f"Silver rows: {silver_check.count():,}")
print(f"Dropped from bronze: {16_095_794 - silver_check.count():,}")
silver_check.select(
    "Covered_Recipient_Profile_ID", "specialty_group", "specialty",
    "Total_Amount_of_Payment_USDollars", "Nature_of_Payment_or_Transfer_of_Value"
).show(5, truncate=False)

Silver rows: 16,095,794
Dropped from bronze: 0



**Step 2 recap.** Bronze 16,095,794 rows → silver, all rows retained (0 dropped, 0 duplicates). 

Specialty split into `specialty_group` (broad type) and `specialty` (specific). 

Written to `silver/clinicians_silver` (Delta, idempotent overwrite).

## Step 3 - Gold layer A: Per-clinician features

Group 16M payment rows to one row per clinician (`Covered_Recipient_Profile_ID`): magnitude, breadth, and identity context.

In [0]:
from pyspark.sql import functions as F
silver_df = spark.read.format("delta").load(silver_path)

# One row per clinician: totals, counts, spread
clinician_features = (
    silver_df.groupBy("Covered_Recipient_Profile_ID")
    .agg(
        #carry the clinician's identity/context (constant across their rows)
        F.first("Covered_Recipient_Type").alias("recipient_type"),
        F.first("specialty_group").alias("specialty_group"),
        F.first("specialty").alias("specialty"),
        F.first("Recipient_State").alias("state"),

        # payment magnitude
        F.sum("Total_Amount_of_Payment_USDollars").alias("total_payment_usd"),
        F.count("*").alias("payment_count"),
        F.avg("Total_Amount_of_Payment_USDollars").alias("avg_payment_usd"),
        F.max("Total_Amount_of_Payment_USDollars").alias("max_payment_usd"),
        F.min("Total_Amount_of_Payment_USDollars").alias("min_amount"),

        # breadth: distinct companies paying this clinician
        F.countDistinct("Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name").alias("distinct_manufacturers"),

        # breadth: distinct payment natures (consulting, food, travel...)
        F.countDistinct("Nature_of_Payment_or_Transfer_of_Value").alias("distinct_nature"),
    )
)

print(f"Distinct Clinicians: {clinician_features.count():,}")
clinician_features.orderBy(F.desc("total_payment_usd")).show(5, truncate=False)

Distinct Clinicians: 1,022,513


**Step 3 recap.** 16M payment rows → **1,022,513 distinct clinicians**. Features: `total/avg/max/min_payment_usd`, `payment_count`, `distinct_manufacturers`, `distinct_nature`. The top of the distribution runs into the tens of millions of dollars from a handful of payments.

## Step 4a - Manufacturer concentration

From silver, compute each clinician's largest single-manufacturer total, then express it as a share of their overall total.
- `top_manufacturer_usd` - $ from the one company that paid them most
- `top_manufacturer_share` - that as a fraction of `total_payment_usd` (0-1)

A share near **1.0** = near-total financial dependence on a single company (a strong COI signal)



In [0]:
from pyspark.sql import functions as F

silver_df = spark.read.format("delta").load(silver_path)

# 1. Summ payments per clinician PER manufacturer
per_mfr = (
    silver_df.groupBy("Covered_Recipient_Profile_ID", "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name")
    .agg(F.sum("Total_Amount_of_Payment_USDollars").alias("mfr_total"))
)

# 2. For each clinician, keep only their single largest manufacturer total
top_mfr = (
    per_mfr.groupBy("Covered_Recipient_Profile_ID")
    .agg(F.max("mfr_total").alias("top_manufacturer_usd"))
)



In [ ]:
# 3. Attach to the per-clinician table and derive the share
clinician_features = (
    clinician_features
    .join(top_mfr, on="Covered_Recipient_Profile_ID", how="left")
    .withColumn(
        "top_manufacturer_share",
        F.round(F.col("top_manufacturer_usd") / F.col("total_payment_usd"), 4)
    )
)

clinician_features.select(
    "Covered_Recipient_Profile_ID", "specialty", "total_payment_usd",
    "distinct_manufacturers", "top_manufacturer_usd", "top_manufacturer_share"
).orderBy(F.desc("top_manufacturer_share")).show(5, truncate=False)

**Step 4a recap.** `top_manufacturer_share = top_manufacturer_usd / total_payment_usd`. Top rows sit at share = 1.0 with `distinct_manufacturers = 1` — entire total from one company. Concentration alone isn't alarming (many such rows are small-dollar); it matters **combined with** magnitude and the specialty benchmark.

## Step 4b - Speciality Peer benchmark
Compare each clinician's `total_payment_usd` against others in the same`specialty`:
- `specialty_avg_total`, `specialty_std_total` - peer group mean & spread
- `payment_zscore_in_specialty` - standardized position vs peers
- `payment_pct_in_specialty` - percentile rank within specialty (0-1)

These reframe raw dollars as *"high **for this specialty**"* - the core anomaly signal

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# peer group = clinicians sharing a specialty
spec_window = Window.partitionBy("specialty")

clinician_features = (
    clinician_features
    .withColumn("specialty_avg_total", F.avg("total_payment_usd").over(spec_window))
    .withColumn("specialty_std_total", F.stddev("total_payment_usd").over(spec_window))
    .withColumn("specialty_clinician_count", F.count("*").over(spec_window))
)

# z-score, guarding against specialties with 0 or null sttdev (single-number groups)
clinician_features = clinician_features.withColumn(
    "payment_zscore_in_specialty",
    F.when(
        (F.col("specialty_std_total").isNotNull()) & (F.col("specialty_std_total") > 0),
        F.round((F.col("total_payment_usd") - F.col("specialty_avg_total")) / F.col("specialty_std_total"), 4)
    ).otherwise(0.0) # lone-clinician specialty → no peers → z = 0
)

In [ ]:
# Percentile rank within specialty (robust to the skew that inflates z-scores)
pct_window = Window.partitionBy("specialty").orderBy("total_payment_usd")

clinician_features = clinician_features.withColumn(
    "payment_pct_in_specialty",
    F.round(F.percent_rank().over(pct_window), 4)
)

clinician_features.select(
    "specialty", "total_payment_usd", "specialty_avg_total",
    "payment_zscore_in_specialty", "payment_pct_in_specialty"
).orderBy(F.desc("payment_zscore_in_specialty")).show(5, truncate=False)

**Step 4b recap**

**What the cells did:**
1. Built a specialty peer group (`Window.partitionBy("specialty")`) and computed each group's mean, stddev, and clinician count.
2. Derived `payment_zscore_in_specialty` — standard deviations above/below the specialty mean — with a guard setting lone-clinician specialties to 0.
3. Derived `payment_pct_in_specialty` — percentile rank within specialty (0–1) via `percent_rank()`.

 Percentile rank kept alongside z-score because the data is heavily right-skewed (the largest totals sit hundreds of standard deviations above their specialty mean). Percentile is the skew-robust view that feeds the model.

## Step 5 - Write gold feature table + validate
Persist the gold table and assert row count, nulls, and range invariants.

In [0]:
# Write gold as Delta 
(clinician_features.write
  .format("delta")
  .mode("overwrite")
  .save(gold_path))

gold_df = spark.read.format("delta").load(gold_path)
print(f"Gold rows written: {gold_df.count()}")
gold_df.printSchema()

Gold rows written: 1022513
root
 |-- Covered_Recipient_Profile_ID: long (nullable = true)
 |-- recipient_type: string (nullable = true)
 |-- specialty_group: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- state: string (nullable = true)
 |-- total_payment_usd: double (nullable = true)
 |-- payment_count: long (nullable = true)
 |-- avg_payment_usd: double (nullable = true)
 |-- max_payment_usd: double (nullable = true)
 |-- min_amount: double (nullable = true)
 |-- distinct_manufacturers: long (nullable = true)
 |-- distinct_nature: long (nullable = true)
 |-- top_manufacturer_usd: double (nullable = true)
 |-- top_manufacturer_share: double (nullable = true)
 |-- specialty_avg_total: double (nullable = true)
 |-- specialty_std_total: double (nullable = true)
 |-- specialty_clinician_count: long (nullable = true)
 |-- payment_zscore_in_specialty: double (nullable = true)
 |-- payment_pct_in_specialty: double (nullable = true)



In [0]:
from pyspark.sql import functions as F

gold_df = spark.read.format("delta").load(gold_path)

# 1. Row count must equal distinct clinicians from Step 3
print(f"Gold rows: {gold_df.count():,}  (expected 1,022,513)")

# 2. Null check on the columns the ML model will consume
feature_cols = [
    "total_payment_usd", "payment_count", "avg_payment_usd",
    "max_payment_usd", "distinct_manufacturers", "distinct_nature",
    "top_manufacturer_share", "payment_zscore_in_specialty",
    "payment_pct_in_specialty",
]

null_report = gold_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in feature_cols
]).collect()[0].asDict()

print("\nNulls in feature columns:")
for c, n in null_report.items():
    flag = "  ⚠️" if n > 0 else ""
    print(f"  {c:<32} {n:>10,}{flag}")

Gold rows: 1,022,513  (expected 1,022,513)

Nulls in feature columns:
  total_payment_usd                         0
  payment_count                             0
  avg_payment_usd                           0
  max_payment_usd                           0
  distinct_manufacturers                    0
  distinct_nature                           0
  top_manufacturer_share                    0
  payment_zscore_in_specialty               0
  payment_pct_in_specialty                  0


In [0]:
# 3. Range sanity — these invariants must hold or a feature is broken
checks = gold_df.select(
    F.min("total_payment_usd").alias("min_total"),          # must be > 0
    F.min("top_manufacturer_share").alias("min_share"),     # must be >= 0
    F.max("top_manufacturer_share").alias("max_share"),     # must be <= 1.0
    F.min("payment_pct_in_specialty").alias("min_pct"),     # must be >= 0
    F.max("payment_pct_in_specialty").alias("max_pct"),     # must be <= 1.0
).collect()[0].asDict()

print("Range checks:")
for k, v in checks.items():
    print(f"  {k:<12} {v}")

# Explicit pass/fail on the invariants
assert checks["min_total"] > 0,        "FAIL: non-positive total payment"
assert checks["max_share"] <= 1.0001,  "FAIL: manufacturer share > 1"
assert checks["max_pct"]   <= 1.0,     "FAIL: percentile > 1"
print("\n✅ All range invariants hold.")

Range checks:
  min_total    0.13
  min_share    0.0508
  max_share    1.0
  min_pct      0.0
  max_pct      1.0

✅ All range invariants hold.


### Silver / Gold layer complete

**Pipeline:** Bronze (16,095,794 rows) → Silver (all retained) → Gold (**1,022,513 clinicians**, `gold/clinician_features`).

| Group | Features |
|---|---|
| Magnitude | `total/avg/max/min_payment_usd`, `payment_count` |
| Breadth | `distinct_manufacturers`, `distinct_nature` |
| Concentration | `top_manufacturer_usd`, `top_manufacturer_share` |
| Specialty benchmark | `specialty_avg/std_total`, `payment_zscore_in_specialty`, `payment_pct_in_specialty` |

**Validation passed:** row count matches distinct clinicians · zero feature nulls · shares & percentiles ∈ [0,1] · totals > 0.

**Feeds three workloads:** Batch KPIs · Structured Streaming · MLlib anomaly detection.

## Step 6 - Batch Processing: Three KPIs
Three single-pass Spark aggregations off the gold layer.

### KPI 1 - Total payment exposure
Total dollars flowing to in-scope clinicians, with record and clinician counts for reconciliation.

In [0]:
from pyspark.sql import functions as F

kpi1 = gold_df.agg(
    F.round(F.sum("total_payment_usd"), 2).alias("total_payment_dollars"),
    F.count("*").alias("paid_clinicians"),
    F.sum("payment_count").alias("total_payment_records"),
)
kpi1.show(truncate=False)

+---------------------+---------------+---------------------+
|total_payment_dollars|paid_clinicians|total_payment_records|
+---------------------+---------------+---------------------+
|2.89488338223E9      |1022513        |16095794             |
+---------------------+---------------+---------------------+



### KPI 2 - Share of clinicians paid materially
Because the dataset contains only clinicians who received payments, the informative ratio is the share exceeding a materiality threshold.

In [0]:
MATERIALITY_THRESHOLD = 5000.0  # analyst-chosen; tune and justify in the writeup

kpi2 = (
    gold_df.agg(
        F.count("*").alias("all_paid_clinicians"),
        F.sum(
            F.when(F.col("total_payment_usd") >= MATERIALITY_THRESHOLD, 1).otherwise(0)
        ).alias("material_clinicians"),
    )
    .withColumn(
        "share_material_pct",
        F.round(F.col("material_clinicians") / F.col("all_paid_clinicians") * 100, 2),
    )
)
kpi2.show(truncate=False)

+-------------------+-------------------+------------------+
|all_paid_clinicians|material_clinicians|share_material_pct|
+-------------------+-------------------+------------------+
|1022513            |43876              |4.29              |
+-------------------+-------------------+------------------+



### KPI 3 - Avg payment per clinician vs. national specialty benchmark
National per-specialty average payment is the benchmark; the KPI is the share of clinicians running above their own specialty.

In [0]:
# National benchmark: mean per-clinician total within each specialty
benchmark = (
    gold_df.groupBy("specialty")
        .agg(
            F.count("*").alias("clinicians"),
            F.round(F.avg("total_payment_usd"), 2).alias("specialty_avg_payment"),
        )
)

# Each clinician vs. their specialty benchmark
kpi3 = (
    gold_df.join(benchmark, on="specialty", how="left")
        .withColumn("above_benchmark", F.col("total_payment_usd") > F.col("specialty_avg_payment"))
        .groupBy("specialty")
        .agg(
            F.first("specialty_avg_payment").alias("specialty_avg_payment"),
            F.count("*").alias("clinicians"),
            F.round(F.avg(F.col("above_benchmark").cast("int")) * 100, 2).alias("share_above_benchmark_pct"),
        )
        .orderBy(F.desc("specialty_avg_payment"))
)
kpi3.show(20, truncate=False)


+--------------------------------------------------+---------------------+----------+-------------------------+
|specialty                                         |specialty_avg_payment|clinicians|share_above_benchmark_pct|
+--------------------------------------------------+---------------------+----------+-------------------------+
|Independent Medical Examiner                      |24464.51             |11        |9.09                     |
|Orthopaedic Surgery                               |23498.6              |24950     |9.1                      |
|Neurological Surgery                              |16891.15             |5539      |11.12                    |
|Medical Genetics                                  |9618.02              |381       |15.49                    |
|Dermatology                                       |9285.99              |12619     |7.7                      |
|Allergy & Immunology                              |8866.33              |2921      |11.19              

**Step 6 recap - three KPIs.**

- **KPI 1 - Total exposure:** **$2.89B** across **1,022,513** clinicians and **16,095,794** records (record count matches the raw dataset exactly → clean medallion aggregation).
- **KPI 2 - Materiality ($5,000):** **43,876 clinicians (4.29%)** clear the threshold - payments are **broad but shallow**.
- **KPI 3 - Vs. specialty benchmark:** share-above-average stays ~6–25% across every specialty (never ~50%), confirming the **right skew** that justifies the downstream z-score and anomaly work.

## Step 7 - Structured Streaming: Single-Manufacturer Conflict-of-Interest Flags
Detect clinicians whose payments from **one manufacturer** cross a threshold within a rolling window. Reads **silver** (transaction grain) replayed as files to simulate a stream; event time on `Date_of_Payment`.

In [0]:
dbutils.fs.rm(f"{BUCKET}/streaming/checkpoints/coi_stream_flags", True)
dbutils.fs.rm(f"{BUCKET}/gold/coi_stream_flags", True)

True

In [0]:
silver_path = f"{BUCKET}/silver/clinicians_silver"   # confirm silver path
landing_path    = f"{BUCKET}/streaming/landing"            # simulated stream source (files)
flags_path      = f"{BUCKET}/gold/coi_stream_flags"        # flagged output (Delta)
checkpoint_path = f"{BUCKET}/streaming/checkpoints/coi_stream_flags"

profile_col = "Covered_Recipient_Profile_ID"
mfr_col     = "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name"  # confirm in your silver
amount_col  = "Total_Amount_of_Payment_USDollars"
date_col    = "Date_of_Payment"

WINDOW_LEN   = "90 days"
WINDOW_SLIDE = "90 days"    # set = WINDOW_LEN for a tumbling window (much lighter state on CE)
WATERMARK    = "30 days"
THRESHOLD    = 10000.0
N_FILES      = 24

In [0]:
from pyspark.sql import functions as F

src = (
    spark.read.format("delta").load(silver_path)
    .select(
        F.col(profile_col).alias("profile_id"),
        F.col(mfr_col).alias("manufacturer"),
        F.col(amount_col).cast("double").alias("amount"),
        F.coalesce(                                         # robust regardless of stored type
            F.to_timestamp(F.col(date_col)),               # already date/timestamp or ISO string
            F.to_timestamp(F.col(date_col), "MM/dd/yyyy"),  # CMS raw string fallback
        ).alias("event_time"),
    )
    .where(F.col("event_time").isNotNull())
)

# range-partition by event_time so the stream flows forward in time (makes the watermark visible)
(src.repartitionByRange(N_FILES, "event_time")
    .write.mode("overwrite").parquet(landing_path))

In [0]:
src_schema = spark.read.parquet(landing_path).schema

stream = (
    spark.readStream
    .schema(src_schema)
    .option("maxFilesPerTrigger", 1)     # one file per trigger = simulated arrival
    .parquet(landing_path)
)

flags = (
    stream
    .withWatermark("event_time", WATERMARK)
    .groupBy(
        F.window("event_time", WINDOW_LEN, WINDOW_SLIDE),
        "profile_id",
        "manufacturer",
    )
    .agg(
        F.sum("amount").alias("window_total"),
        F.count("*").alias("payment_count"),
    )
    .where(F.col("window_total") >= THRESHOLD)
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "profile_id", "manufacturer",
        F.round("window_total", 2).alias("window_total"),
        "payment_count",
    )
)

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "16")   

In [ ]:
q = (
    flags.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start(flags_path)
)
# availableNow processes all data then stops — poll instead of blocking forever
while q.isActive:
    q.awaitTermination(timeout=10)   # returns every 10s; loop exits when query self-stops
print("Stream drained. Rows written this run:", q.lastProgress)

In [ ]:
flagged = spark.read.format("delta").load(flags_path)
print("flagged (clinician × manufacturer × window):", flagged.count())
display(flagged.orderBy(F.col("window_total").desc()).limit(25))

**Step 7 recap - Structured Streaming.**

- **Source:** silver payment rows, replayed one file per trigger (`maxFilesPerTrigger=1`).
- **Event time:** `Date_of_Payment`; **30-day watermark** for late payments.
- **Window:** 90-day tumbling, keyed `profile_id × manufacturer`; **flag rule:** window total ≥ **$10,000**.
- **Sink:** `gold/coi_stream_flags` (Delta, append), `trigger(availableNow=True)`.

Flags concentrate in device/orthopedic makers (Zimmer Biomet, Medtronic, DePuy, Arthrex) — consistent with high-payment specialties.

## Stage 8 - MLlib anomaly detection

**Goal:** flag clinicians whose payment patterns are unusual *relative to specialty peers*.

**Approach:** unsupervised K-means on two peer-relative features. Distance from the *normal* cluster centroid = anomaly score. No ground-truth COI labels exist, so evaluation = clustering quality (silhouette) + face validity, **not** precision/recall.

### Step 8.1 - Feature assembly
Two features, both already bounded 0–1: `payment_pct_in_specialty` (rank vs peers) and `top_manufacturer_share` (concentration). No scaling needed. Magnitude columns are deliberately excluded so clustering keys on peer-deviation, not raw wealth (absolute dollars re-enter only as a shortlist tiebreaker).

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Peer-relative signals only — both already 0-1, no transforms needed
feature_cols = [
    "payment_pct_in_specialty",   # where you rank vs peers in your specialty
    "top_manufacturer_share",     # how concentrated on one company
]

# Select only ID + feature columns to reduce DataFrame size
gold_subset = clinician_features.select(
    "Covered_Recipient_Profile_ID",
    "specialty",
    "total_payment_usd",
    "payment_count",
    *feature_cols
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip",
)
gold_scaled = assembler.transform(gold_subset)

before = gold_subset.count()
after  = gold_scaled.count()
print(f"Rows in: {before:,} | after assemble: {after:,} | dropped: {before - after:,}")
print("Note: Both features are already 0-1, so no scaling needed for K-means")

gold_scaled.select("Covered_Recipient_Profile_ID", "features").show(5, truncate=False)

Rows in: 1,022,513 | after assemble: 1,022,513 | dropped: 0
Note: Both features are already 0-1, so no scaling needed for K-means


**What the cell did:** combined the two peer-relative features into one vector. No scaling: both are already 0–1.

| Feature | Meaning | Range |
|---|---|---|
| `payment_pct_in_specialty` | Percentile rank vs. specialty peers | 0–1 |
| `top_manufacturer_share` | Share of payments from one company | 0–1 |

- `VectorAssembler` → `features` (no `StandardScaler`, which also kept the serverless model cache small)
- **0 rows dropped** - all 1,022,513 clinicians retained
- No log/skew correction needed: both features are already bounded 0–1

**Why only two features:** these two encode the COI question directly — *unusual for your specialty* + *concentrated on one company*. Magnitude columns (total/max/count) were deliberately left out so clustering keys on peer-deviation, not raw wealth; absolute dollars re-enter later only as a shortlist tiebreaker.

### Step 8.2 - Fit K-Means + Choose k

Sweep k = 2–8, scoring each with silhouette; `del` each fit to keep the serverless driver ML cache under its cap.

In [0]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

print("k | silhouette")
print("-----------------")
for k in range(2, 9):
    km = KMeans(featuresCol="features", k=k, seed=42)
    m = km.fit(gold_scaled)
    score = evaluator.evaluate(m.transform(gold_scaled))
    print(f"{k} | {score:.4f}")
    del m, km          # release each fit from the driver cache before the next

k | silhouette
-----------------
2 | 0.6526
3 | 0.6478
4 | 0.6771
5 | 0.6521
6 | 0.6214
7 | 0.6316
8 | 0.6092


**Recap.** Best silhouette at **k = 4 (0.677)**; all k score 0.61–0.68 (strong separation). Four clusters map onto the two axes (specialty rank × concentration).


### Step 8.3 - Fit final model + anomaly score

Fit k=4, auto-detect the **normal** cluster (lowest combined pct + share), and score each clinician by distance from that baseline centroid. High rank and/or high concentration = far from normal = higher priority. (Centroids passed as plain Python lists - serverless blocks `sparkContext.broadcast`.)

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans

# --- Fit final model at k=4 ---
kmeans = KMeans(featuresCol="features", k=4, seed=42)
model = kmeans.fit(gold_scaled)
clustered = model.transform(gold_scaled)

# --- 1) Profile each cluster to find the "normal" (low pct + low share) group ---
profile = (clustered.groupBy("prediction")
    .agg(F.count("*").alias("n"),
         F.round(F.mean("payment_pct_in_specialty"), 3).alias("avg_pct"),
         F.round(F.mean("top_manufacturer_share"), 3).alias("avg_share"))
    .orderBy("prediction"))
profile.show()

normal_id = (profile
    .withColumn("risk_sum", F.col("avg_pct") + F.col("avg_share"))
    .orderBy("risk_sum").first()["prediction"])
print("Normal (baseline) cluster:", normal_id)

# --- 2) Anomaly score = distance from the NORMAL centroid ---
normal_centroid = model.clusterCenters()[normal_id].tolist()

@F.udf("double")
def dist_from_normal(features):
    import math
    return float(math.sqrt(sum((f - c) ** 2 for f, c in zip(features, normal_centroid))))

scored = clustered.withColumn("anomaly_score", dist_from_normal(F.col("features")))

# --- 3) Top anomalies — should be HIGH pct and/or HIGH share ---
scored.select(
    "Covered_Recipient_Profile_ID", "specialty",
    "payment_pct_in_specialty", "top_manufacturer_share",
    "prediction", "anomaly_score",
).orderBy(F.desc("anomaly_score")).show(10, truncate=False)

+----------+------+-------+---------+
|prediction|     n|avg_pct|avg_share|
+----------+------+-------+---------+
|         0|179484|  0.761|    0.879|
|         1|281216|  0.775|    0.316|
|         2|327607|    0.2|    0.989|
|         3|234206|  0.388|     0.52|
+----------+------+-------+---------+

Normal (baseline) cluster: 3



**Recap:** fit the final K-means (k=4), then scored every clinician by distance from the **normal** cluster's centroid - the low-rank, low-concentration baseline.

- **Score = distance from "typical clinician"** — high specialty rank and/or high single-company concentration = far from normal = higher COI priority
- Normal cluster identified automatically as the one with lowest combined avg pct + share (came out as cluster 3)
- Centroids handled as plain Python lists inside the UDF (serverless blocks `sparkContext.broadcast`)

**Face-validity check passes:** top anomalies are all `payment_pct_in_specialty` = 1.0 **and** `top_manufacturer_share` = 1.0 - exactly the clinicians worth investigating.


### Step 8.3b - Name the Clusters

The four clusters on the rank × concentration grid. **Labels are keyed to this run's cluster IDs** - the `groupBy` below verifies each ID against its profile.

In [ ]:
from pyspark.sql import functions as F

# Name each cluster from where its centroid sits, not from its ID.
# K-means IDs can renumber between fits, so a hard-coded {id: name} map can silently go wrong.
def name_cluster(pct, share):
    if pct >= 0.5 and share >= 0.7:  return "Concentrated High-Rank", "HIGH"
    if pct <  0.5 and share >= 0.7:  return "Concentrated Low-Rank",  "MEDIUM"
    if pct >= 0.5 and share <  0.7:  return "Diversified High-Rank",  "MEDIUM"
    return "Baseline", "LOW"

CLUSTER_LABELS, CLUSTER_RISK = {}, {}
for i, c in enumerate(model.clusterCenters()):
    CLUSTER_LABELS[i], CLUSTER_RISK[i] = name_cluster(c[0], c[1])
    print(f"Cluster {i}: pct={c[0]:.3f}, share={c[1]:.3f} -> {CLUSTER_LABELS[i]} ({CLUSTER_RISK[i]})")

label_expr = F.lit("Unknown")
for i, name in CLUSTER_LABELS.items():
    label_expr = F.when(F.col("prediction") == i, name).otherwise(label_expr)
scored = scored.withColumn("cluster_label", label_expr)

scored.groupBy("prediction", "cluster_label").count().orderBy("prediction").show(truncate=False)


**Cluster map.** Names come from each centroid's position: high or low percentile (cut at 0.5) crossed with concentrated or diversified (cut at 0.7 share). In the final run: **Concentrated High-Rank** (0.761, 0.879) · **Diversified High-Rank** (0.775, 0.317) · **Concentrated Low-Rank** (0.200, 0.989) · **Baseline** (0.388, 0.520). The baseline matches the cluster picked as "normal" in Step 8.3.


### Step 8.4 - Investigation Shortlist

Rank by anomaly score, tie-break by absolute payment size, take the top 500, persist to gold for Power BI. A triage list - a ranked starting point for review, **not** a determination of wrongdoing.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TOP_N = 500   # size of the investigation shortlist — tune to taste

rank_w = Window.orderBy(F.desc("anomaly_score"), F.desc("total_payment_usd"))

shortlist_out = (scored
    .orderBy(F.desc("anomaly_score"), F.desc("total_payment_usd"))
    .limit(TOP_N)                                    # cut to top N FIRST (cheap)
    .withColumn("investigation_rank", F.row_number().over(rank_w))
    .select(
        "investigation_rank",
        "Covered_Recipient_Profile_ID",
        "specialty",
        "total_payment_usd",
        "payment_count",
        "top_manufacturer_share",
        "payment_pct_in_specialty",
        "anomaly_score",
        "prediction",
        "cluster_label", 
    )
    .orderBy("investigation_rank"))

shortlist_out.show(20, truncate=False)

# --- Persist to gold for Power BI ---
(shortlist_out.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")   # <-- allow the new schema (adds cluster_label)
    .save(f"{BUCKET}/gold/coi_investigation_shortlist"))

print(f"Wrote top {TOP_N} to gold/coi_investigation_shortlist")

**Recap.** Top 500 written to `gold/coi_investigation_shortlist`. Score saturates at the max for anyone at pct=1.0 & share=1.0, so absolute dollars order the top tier. All 500 are Concentrated High-Rank.

**Evaluation summary:** silhouette 0.677 (clustering quality) + face validity (top of list) constitute legitimate evaluation in the absence of ground-truth COI labels; precision/recall does not apply to unsupervised anomaly detection.

## Stage 9 - Model deployment

Stage 8 *built and validated* the model on the CMS population. This stage *deploys* it: a reusable scoring function that takes a clinician's raw annual payment details, computes both model features, and assigns a cluster + risk tier — then applies it to **Mephisto Health System's own roster**.

This is the train/inference split of real ML deployment: CMS PY2025 trained the model; new clinicians are scored against it. `payment_pct_in_specialty` is ranked against the cached CMS specialty distribution (`specialty_totals`), so percentiles stay grounded in the training population.

**Risk tiers** (analyst-defined): Concentrated High-Rank = **HIGH**; Concentrated Low-Rank & Diversified High-Rank = **MEDIUM**; Baseline = **LOW**.

### Step 9.1 - Scoring function + interactive calculator
One Spark pass caches the specialty distributions; after that, scoring is instant in-memory math. Each score cross-checks the Python nearest-centroid result against the trained Spark model and only trusts a match. Widgets at the top of the notebook drive the interactive calculator.

In [0]:
import numpy as np
from pyspark.ml.linalg import Vectors

# --- One-time Spark pass: cache specialty distributions in driver memory ---
_ref = clinician_features.select("specialty", "total_payment_usd").toPandas()
specialty_totals = {s: np.sort(g["total_payment_usd"].values)
                    for s, g in _ref.groupby("specialty")}
valid_specialties = sorted(specialty_totals)

# --- Model artifacts as plain Python (labels derived from centroids in Step 8.3b) ---
centroids = [c.tolist() for c in model.clusterCenters()]


def classify_clinician(specialty, annual_total, biggest_company, validate=True):
    # validation
    if specialty not in specialty_totals:
        return {"error": f"Unknown specialty '{specialty}'."}
    if annual_total <= 0:
        return {"error": "Annual total must be greater than 0."}
    if not 0 <= biggest_company <= annual_total:
        return {"error": "Biggest-company amount must be between 0 and the annual total."}

    # two features
    arr = specialty_totals[specialty]
    pct = min(np.searchsorted(arr, annual_total, "left") / (len(arr) - 1), 1.0) if len(arr) > 1 else 1.0
    share = biggest_company / annual_total
    feats = [float(pct), share]

    # nearest-centroid (mirrors model.predict), then Spark cross-check
    cid = int(np.argmin([np.linalg.norm(np.subtract(feats, c)) for c in centroids]))
    check = "(not validated)"
    if validate:
        spark_cid = int(model.transform(
            spark.createDataFrame([(Vectors.dense(feats),)], ["features"])
        ).first()["prediction"])
        if spark_cid == cid:
            check = "✓ Python matches trained model"
        else:
            cid, check = spark_cid, f"✗ mismatch — model says {spark_cid}, trusting model"

    return {"specialty": specialty, "annual_total": round(annual_total, 2),
            "pct_in_specialty": round(float(pct), 4), "top_share": round(share, 4),
            "cluster_label": CLUSTER_LABELS[cid], "risk_tier": CLUSTER_RISK[cid],
            "validated": check}


# --- Interactive input fields (appear in the widget bar at the top) ---
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("specialty", valid_specialties[0], valid_specialties, "Specialty")
dbutils.widgets.text("annual_total_usd", "0", "Annual total ($)")
dbutils.widgets.text("biggest_company_usd", "0", "Biggest single-company ($)")

print(f"Ready — {len(valid_specialties)} specialties cached. Set the widgets above, then run the score cell.")

Ready — 57 specialties cached. Set the widgets above, then run the score cell.


In [0]:
r = classify_clinician(
    dbutils.widgets.get("specialty"),
    float(dbutils.widgets.get("annual_total_usd")),
    float(dbutils.widgets.get("biggest_company_usd")),
)

print("=" * 52)
if "error" in r:
    print("  ERROR:", r["error"])
else:
    print(f"  {r['specialty']}  |  ${r['annual_total']:,.2f}/yr")
    print(f"  Specialty percentile : {r['pct_in_specialty']:.1%}")
    print(f"  Top-company share    : {r['top_share']:.1%}")
    print("-" * 52)
    print(f"  CLUSTER : {r['cluster_label']}")
    print(f"  RISK    : {r['risk_tier']}")
    print(f"  {r['validated']}")
print("=" * 52)

  Denturist  |  $80,000.00/yr
  Specialty percentile : 100.0%
  Top-company share    : 100.0%
----------------------------------------------------
  CLUSTER : Concentrated High-Rank
  RISK    : HIGH
  ✓ Python matches trained model


**Recap.** `classify_clinician(specialty, annual_total, biggest_company)` → cluster + risk tier, self-validated against the model. Percentiles reference the fixed PY2025 population; cluster assignment is nearest-centroid, so boundary cases follow geometry — expected for hard clustering.

### Step 9.2 - Export centroids for the Power BI calculator
The four centroids + label/risk mapping, so the Power BI page can reproduce scoring in DAX.

In [ ]:
# Extract the four centroids + their labels + risk mapping for the Power BI calculator
for i, c in enumerate(model.clusterCenters()):
    print(f"Cluster {i}: pct={c[0]:.6f}, share={c[1]:.6f}  "
          f"-> {CLUSTER_LABELS[i]} ({CLUSTER_RISK[i]})")

### Step 9.3 - Score the Mephisto roster
Read Mephisto's raw roster from `bronze/mephisto_roster.csv` (raw input lands in bronze), score every clinician through the calculator, and write the named, risk-tiered roster to `gold/mephisto_scored` for the dashboard. Specialties must match the CMS data exactly.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType)

# --- Read Mephisto's raw roster from S3 (bronze = raw input) ---
roster = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BUCKET}/bronze/mephisto_roster.csv"))

print(f"Loaded {roster.count()} clinicians from bronze/mephisto_roster.csv")
roster.show(truncate=False)

Loaded 24 clinicians from bronze/mephisto_roster.csv
+-----------------+--------------------------------------------------+----------------+-------------------+
|clinician_name   |specialty                                         |annual_total_usd|biggest_company_usd|
+-----------------+--------------------------------------------------+----------------+-------------------+
|Dr. Alan Reyes   |Orthopaedic Surgery                               |1850000         |1850000            |
|Dr. Bianca Osei  |Neurological Surgery                              |920000          |910000             |
|Dr. Carla Mwangi |Internal Medicine                                 |640000          |638000             |
|Dr. David Kim    |Thoracic Surgery (Cardiothoracic Vascular Surgery)|410000          |405000             |
|Dr. Elena Petrova|Dermatology                                       |88000           |87500              |
|Dr. Farid Nasser |Psychiatry & Neurology                            |52000        

In [0]:
# --- Score each row via the Step 9.1 calculator (reuses feature calc + model validation) ---
scored_rows = []
for row in roster.collect():
    r = classify_clinician(row["specialty"],
                           float(row["annual_total_usd"]),
                           float(row["biggest_company_usd"]),
                           validate=True)
    if "error" in r:
        print(f"  SKIPPED {row['clinician_name']}: {r['error']}")
        continue
    scored_rows.append((
        row["clinician_name"], r["specialty"], float(r["annual_total"]),
        float(row["biggest_company_usd"]), r["pct_in_specialty"], r["top_share"],
        r["cluster_label"], r["risk_tier"],
    ))

print(f"Scored {len(scored_rows)} of {roster.count()} clinicians.")

Scored 24 of 24 clinicians.


In [ ]:
# --- Build DataFrame, verify tier spread, write to gold ---
schema = StructType([
    StructField("clinician_name",           StringType(), False),
    StructField("specialty",                StringType(), False),
    StructField("annual_total_usd",         DoubleType(), False),
    StructField("biggest_company_usd",      DoubleType(), False),
    StructField("payment_pct_in_specialty", DoubleType(), False),
    StructField("top_manufacturer_share",   DoubleType(), False),
    StructField("cluster_label",            StringType(), False),
    StructField("risk_tier",                StringType(), False),
])
mephisto_df = spark.createDataFrame(scored_rows, schema)

print("Risk tier spread:")
mephisto_df.groupBy("risk_tier").count().orderBy("risk_tier").show()
mephisto_df.orderBy("risk_tier", "clinician_name").show(truncate=False)

(mephisto_df.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .save(f"{BUCKET}/gold/mephisto_scored"))
print("Wrote gold/mephisto_scored")

**Recap - model deployed.** Mephisto's roster scored to `gold/mephisto_scored`, spanning HIGH / MEDIUM / LOW (8 / 8 / 8 in the final run). Register as an external table and point the Power BI Mephisto page at it:

```sql
CREATE TABLE IF NOT EXISTS workspace.default.mephisto_scored
USING DELTA
LOCATION 's3://<your-bucket>/gold/mephisto_scored';
```

The four analytical pages show how the model was built; this page shows it deployed on the client's roster - the product story.